# 04 — Test the FE Endpoint

Three checks against the deployed `claims-fe-transformer` endpoint:

1. **Local parity**: load model with `mlflow.pyfunc.load_model()`, run against all
   synthetic payloads, assert every documented `features_json` key is present, no
   extraction errors, and marker-injected payloads fire their respective flags.
2. **Endpoint parity**: query the serving endpoint with the same payloads and
   compare to local output.
3. **Baseline latency**: fire 1 / 10 / 50 concurrent requests via
   `ThreadPoolExecutor`, report p50/p95/p99/error rate. Not a full load test — just
   a first data point for the Model-Serving-vs-Apps decision.

In [0]:
%pip install mlflow /Volumes/fins_genai/classic_ml/claims_fe_wheels/claims_fe-0.1.0-py3-none-any.whl --force-reinstall
dbutils.library.restartPython()

Processing /Volumes/fins_genai/classic_ml/claims_fe_wheels/claims_fe-0.1.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 134.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 144.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [0]:
CATALOG = "fins_genai"
SCHEMA = "classic_ml"
TABLE_NAME = "fe_test_payloads"
REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.claims_fe_transformer"
ENDPOINT_NAME = "claims-fe-transformer"

## Load synthetic payloads

In [0]:
rows = (
    spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")
    .select("payload_id", "lob_type", "loss_cause", "markers_injected", "payload_json")
    .orderBy("payload_id")
    .collect()
)
print(f"Loaded {len(rows)} synthetic payloads")

Loaded 50 synthetic payloads


## 1. Local parity — run pyfunc in-process

In [0]:
import json
import mlflow
import pandas as pd

mlflow.set_registry_uri("databricks-uc")

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
versions = list(w.model_versions.list(full_name=REGISTERED_MODEL_NAME))
latest_version = str(max(int(v.version) for v in versions))
model_uri = f"models:/{REGISTERED_MODEL_NAME}/{latest_version}"
print(f"Loading {model_uri}")

local_model = mlflow.pyfunc.load_model(model_uri)

# Run against all payloads in one batch
batch_in = pd.DataFrame({"payload_json": [r.payload_json for r in rows]})
batch_out = local_model.predict(batch_in)
local_features = [json.loads(x) for x in batch_out["features_json"].tolist()]
print(f"Local pyfunc produced {len(local_features)} feature rows")

Loading models:/fins_genai.classic_ml.claims_fe_transformer/5


/local_disk0/.ephemeral_nfs/envs/pythonEnv-2a832a34-438f-406e-a429-b66adc3f4bee/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Local pyfunc produced 50 feature rows


### Contract assertions

In [0]:
REQUIRED_KEYS = {
    "claim_id", "claim_num", "lob_type", "loss_cause", "sub_type", "state",
    "jurisdiction_state", "water_source",
    "loss_date", "reported_date", "days_loss_to_report", "property_age_at_loss",
    "n_exposures", "has_dwelling", "has_contents", "has_living_expenses", "exposure_types",
    "desc_char_count", "docs_char_count", "notes_char_count", "incidents_char_count",
    "attorney_mentioned", "attorney_phrase_count", "siu_red_flag_count",
    "medical_severity_flag", "subrogation_opportunity_flag", "health_disclosure_flag",
    "urgency_score", "max_dollar_amount_mentioned", "dollar_amount_count",
    "feature_version", "extraction_errors",
}

for r, feats in zip(rows, local_features):
    missing = REQUIRED_KEYS - feats.keys()
    assert not missing, f"{r.payload_id}: missing keys {missing}"
    assert feats["extraction_errors"] == [], f"{r.payload_id}: errors {feats['extraction_errors']}"

print(f"All {len(rows)} payloads pass key-completeness + no-errors check.")

All 50 payloads pass key-completeness + no-errors check.


### Marker-injection fires the right flags

For each injected marker, the corresponding flag must be True / positive.

In [0]:
MARKER_TO_FLAG = {
    "attorney":    ("attorney_mentioned", True),
    "siu":         ("siu_red_flag_count", "> 0"),
    "medical":     ("medical_severity_flag", True),
    "subrogation": ("subrogation_opportunity_flag", True),
    "health":      ("health_disclosure_flag", True),
    "dollar":      ("dollar_amount_count", "> 0"),
}

failures = []
for r, feats in zip(rows, local_features):
    for marker in r.markers_injected:
        if marker not in MARKER_TO_FLAG:
            continue
        key, expected = MARKER_TO_FLAG[marker]
        actual = feats.get(key)
        ok = (actual is True) if expected is True else (isinstance(actual, (int, float)) and actual > 0)
        if not ok:
            failures.append(f"{r.payload_id} marker={marker} -> {key}={actual} (expected {expected})")

if failures:
    print("MARKER PARITY FAILURES:")
    for f in failures:
        print(f"  {f}")
    raise AssertionError(f"{len(failures)} marker-flag mismatches")
print(f"All marker injections light up the expected flags.")

All marker injections light up the expected flags.


## 2. Endpoint parity

Requires the endpoint to be READY.

In [0]:
ep = w.serving_endpoints.get(ENDPOINT_NAME)
print(f"Endpoint state: {ep.state.ready}")

Endpoint state: EndpointStateReady.READY


In [ ]:
# Pull from a Databricks secret scope or env var — never hard-code.
import os
SP_CLIENT_ID = os.environ.get("SP_CLIENT_ID", "<your-service-principal-client-id>")
SP_CLIENT_SECRET = os.environ.get("SP_CLIENT_SECRET", "<your-service-principal-client-secret>")

In [0]:
import databricks.sdk.core as client

# Route-optimized endpoints require OAuth auth (service principal).
# Set SP_CLIENT_ID and SP_CLIENT_SECRET via secrets or widgets before running.
c = client.Config(
    host=w.config.host,
    client_id=SP_CLIENT_ID, #pull from your secret or env
    client_secret=SP_CLIENT_SECRET, #pull from your secret or env
)
w_oauth = WorkspaceClient(config=c)

def query_endpoint(payload_json_values):
    records = [{"payload_json": v} for v in payload_json_values]
    resp = w_oauth.serving_endpoints_data_plane.query(name=ENDPOINT_NAME, dataframe_records=records)
    return [json.loads(p["features_json"]) for p in resp.predictions]


endpoint_features = query_endpoint([r.payload_json for r in rows])

# Compare local vs endpoint byte-for-byte (keys / values)
mismatches = []
for r, local, remote in zip(rows, local_features, endpoint_features):
    if local != remote:
        diff_keys = [k for k in set(local) | set(remote) if local.get(k) != remote.get(k)]
        mismatches.append((r.payload_id, diff_keys))

if mismatches:
    print("Mismatches:")
    for pid, keys in mismatches[:10]:
        print(f"  {pid}: {keys}")
    raise AssertionError(f"{len(mismatches)} endpoint/local mismatches")
print(f"All {len(rows)} payloads match between local pyfunc and endpoint response.")

All 50 payloads match between local pyfunc and endpoint response.


## 3. Baseline latency probe

Fire N concurrent single-row requests. Not a full load test — just a sanity reading
against the provisioned endpoint. Interpret cautiously: this measures end-to-end
client→endpoint→client time, not just `predict()`.

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from statistics import quantiles, median

payloads = [r.payload_json for r in rows]


def single_request(payload_json):
    t0 = time.perf_counter()
    try:
        w_oauth.serving_endpoints_data_plane.query(
            name=ENDPOINT_NAME,
            dataframe_records=[{"payload_json": payload_json}],
        )
        return ("ok", time.perf_counter() - t0)
    except Exception as exc:
        return (f"err:{type(exc).__name__}", time.perf_counter() - t0)


def probe(concurrency: int, n_requests: int = 100):
    results = []
    import random
    wall_start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(single_request, random.choice(payloads)) for _ in range(n_requests)]
        for fut in as_completed(futures):
            results.append(fut.result())
    wall_secs = time.perf_counter() - wall_start
    latencies = sorted(r[1] * 1000 for r in results)  # ms
    errs = sum(1 for r in results if not r[0].startswith("ok"))
    ok_count = n_requests - errs
    qps = ok_count / wall_secs if wall_secs > 0 else 0
    qs = quantiles(latencies, n=20) if len(latencies) >= 20 else [0] * 19
    print(
        f"concurrency={concurrency:>3}  n={n_requests}  "
        f"p50={median(latencies):7.1f}ms  "
        f"p95={qs[18] if len(qs) > 18 else max(latencies):7.1f}ms  "
        f"max={max(latencies):7.1f}ms  "
        f"QPS={qps:6.1f}  "
        f"errors={errs}"
    )


# Warm-up
for _ in range(3):
    single_request(payloads[0])

print("Baseline latency probe:")
probe(concurrency=1, n_requests=30)
probe(concurrency=10, n_requests=100)
probe(concurrency=50, n_requests=200)

Baseline latency probe:
concurrency=  1  n=30  p50=   31.6ms  p95=   39.0ms  max=   43.0ms  QPS=  32.1  errors=0
concurrency= 10  n=100  p50=   64.9ms  p95=  229.1ms  max=  268.0ms  QPS= 126.0  errors=0
concurrency= 50  n=200  p50=  144.4ms  p95=  525.2ms  max=  794.7ms  QPS= 156.0  errors=0


## Interpretation

- p50 / p95 at concurrency=1 gives you the per-request overhead of the endpoint.
- p95 growing sharply between concurrency=10 and concurrency=50 indicates worker
  saturation at the current workload size — scale up `workload_size` or switch to
  the Databricks Apps option (per the Google Doc's decision matrix).
- Error rate > 0 at higher concurrency usually means request queue timeout or
  body-size rejection. Check the endpoint logs via Databricks UI.